In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import linear_kernel

In [2]:
data = pd.read_csv('https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Data/unsupervised_sprint/netflix_titles.csv', index_col=0)
data.head()

,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
show_id,,,,,,,,,,,
81145628,Movie,Norm of the North: King Sized Adventure,"Richard Finn, Tim Maltby","Alan Marriott, Andrew Toth, Brian Dobson, Cole...","United States, India, South Korea, China","September 9, 2019",2019,TV-PG,90 min,"Children & Family Movies, Comedies",Before planning an awesome wedding for his gra...
80117401,Movie,Jandino: Whatever it Takes,NaN,Jandino Asporaat,United Kingdom,"September 9, 2016",2016,TV-MA,94 min,Stand-Up Comedy,Jandino Asporaat riffs on the challenges of ra...
70234439,TV Show,Transformers Prime,NaN,"Peter Cullen, Sumalee Montano, Frank Welker, J...",United States,"September 8, 2018",2013,TV-Y7-FV,1 Season,Kids' TV,"With the help of three human allies, the Autob..."
80058654,TV Show,Transformers: Robots in Disguise,NaN,"Will Friedle, Darren Criss, Constance Zimmer, ...",United States,"September 8, 2018",2016,TV-Y7,1 Season,Kids' TV,When a prison ship crash unleashes hundreds of...
80125979,Movie,#realityhigh,Fernando Lebrija,"Nesta Cooper, Kate Walsh, John Michael Higgins...",United States,"September 8, 2017",2017,TV-14,99 min,Comedies,When nerdy high schooler Dani finally attracts...


In [3]:
# Drop rows with NaN values in the specified columns to ensure we have complete data for these important attributes
data.dropna(subset=['cast', 'title', 'description', 'listed_in'], inplace=True, axis=0)

# Reset the index after dropping rows to maintain a clean DataFrame
data = data.reset_index(drop=True)

# Clean text data by removing punctuation and extra spaces to ensure consistency in the text data
data['listed_in'] = [re.sub(r'[^\w\s]', '', t) for t in data['listed_in']] 
data['cast'] = [re.sub(',', ' ', re.sub(' ', '', t)) for t in data['cast']] 
data['description'] = [re.sub(r'[^\w\s]', '', t) for t in data['description']]
data['title'] = [re.sub(r'[^\w\s]', '', t) for t in data['title']] 

# Combine the cleaned text data into a single 'combined' feature
data["combined"] = data['listed_in'] + ' ' + data['cast'] + ' ' + data['title'] + ' ' + data['description']

# Drop the individual columns as they are now represented in the 'combined' feature
data.drop(['director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'cast', 'description'], axis=1, inplace=True)

# Display the first few rows of the cleaned data to verify the changes
data.head()

,type,title,combined
0,Movie,Norm of the North King Sized Adventure,Children Family Movies Comedies AlanMarriott ...
1,Movie,Jandino Whatever it Takes,StandUp Comedy JandinoAsporaat Jandino Whateve...
2,TV Show,Transformers Prime,Kids TV PeterCullen SumaleeMontano FrankWelker...
3,TV Show,Transformers Robots in Disguise,Kids TV WillFriedle DarrenCriss ConstanceZimme...
4,Movie,realityhigh,Comedies NestaCooper KateWalsh JohnMichaelHigg...


In [4]:
# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the combined text data to numerical vectors
matrix = vectorizer.fit_transform(data["combined"])

# Compute cosine similarities between the vectors
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarities = cosine_similarity(matrix, matrix)

# Extract the title column for later use in recommendations
movie_title = data['title']

# Create a series to map each title to its index in the dataset
indices = pd.Series(data.index, index=data['title'])

In [5]:
# Define the recommendation function
def content_recommender(title):
    # Get the index of the given title
    idx = indices[title]

    # Get the pairwise similarity scores of all titles with the given title
    sim_scores = list(enumerate(cosine_similarities[idx]))

    # Sort the titles based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the indices of the most similar titles
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]

    # Return the titles of the most similar titles
    return movie_title.iloc[movie_indices]

In [6]:
# Test the recommendation function
content_recommender('The Crown')

369                         Witches A Century of Murder
1829                                         London Spy
5068                                              Reign
2612                                     My Hotter Half
692     The Blue Planet A Natural History of the Oceans
3915                        The Real Football Factories
1753                                         Collateral
5474                                           Lovesick
4830               Planet Earth The Complete Collection
2724                                       Age Gap Love
Name: title, dtype: object